In [1]:
import torch
import time
import keyboard
from IPython.display import clear_output

from camera_wrapper import CameraWrapper
from hand_tracker import HandTracker
from online_k_means import OnlineKmeans
from live_dataset import LiveDataset
from autoencoder import HumanSignalAutoencoder
from ae_trainer import AETrainer
from hand_visualizer import HandVisualizer
from centroid_labeler import CentroidLabeler
from label_getter import LabelGetter

In [2]:
# settings

In [3]:
# architecture settings
human_signal_shape = (1, 21, 3) # 1 hand * 21 tracking points * 3 tracking dims
autoencoder_bottleneck = 32

# region classifier settings
okm_num_centroids = 5

# live dataset settings
queue_size_per_class = 200

# autoenc training settings
lr = 0.0001
ae_batch_size = 32
ae_steps_per_frame = 10

# label settings
label_keys = ['w', 'a', 's', 'd', 'shift']
num_labels = len(label_keys)

# maps each label to the key that gets pressed in auto-press mode (None = no action)
action_map = {
    'w': 'w',
    'a': 'a',
    's': 's',
    'd': 'd',
    'shift': None,
}

# label assignment confidence gate:
# nearest_dist / second_nearest_dist must be below this to store a label.
# lower = stricter. 1.0 = no gate.
label_margin_threshold = 0.75

# control keys
quit_key          = 'q'
autopress_toggle_key = 'space'

# torch settings
run_device = torch.device("cuda")

In [4]:
camera = CameraWrapper(camera_index=1)
hand_tracker = HandTracker("./hand_landmarker.task")

In [5]:
true_visualizer = HandVisualizer('True Hand')
decoded_visualizer = HandVisualizer('Decoded Hand')

In [6]:
autoenc = HumanSignalAutoencoder(human_signal_shape, autoencoder_bottleneck).to(run_device)
region_classifier = OnlineKmeans(okm_num_centroids, run_device)
live_dataset = LiveDataset(okm_num_centroids, queue_size_per_class)
ae_trainer = AETrainer(autoenc, lr=lr, device=run_device)
centroid_labeler = CentroidLabeler(okm_num_centroids, num_labels)
label_getter = LabelGetter(label_keys)

In [ ]:
start_time = time.time()
autopress_enabled = False
toggle_was_pressed = False
active_key = None
label_map = {i: None for i in range(okm_num_centroids)}
confidence_map = {i: 0.0 for i in range(okm_num_centroids)}
loss_val = float('nan')

while not keyboard.is_pressed(quit_key):

    # capture frame
    img = camera()

    # track hand — skip frame if shape doesn't match (0 or 2+ hands)
    human_signal = hand_tracker(img, int((time.time() - start_time) * 1000)).to(run_device)
    if human_signal.shape != torch.Size(human_signal_shape):
        continue

    # encode
    with torch.no_grad():
        latent = autoenc.encode(human_signal.unsqueeze(0)).squeeze(0)

    # classify
    class_idx, dists = region_classifier.classify(latent)
    if class_idx is None:
        class_idx = 0  # OKM not yet initialised — default to class 0

    # get ground truth label (None if user isn't holding a label key)
    label = label_getter.get_label()
    label_key_pressed = label is not None
    if label is not None and dists is not None:
        sorted_dists = dists.sort().values
        margin = sorted_dists[0] / (sorted_dists[1] + 1e-8)
        if margin >= label_margin_threshold:
            label = None  # assignment too ambiguous — withhold label

    # store datapoint
    live_dataset.apply_datapoint(latent, human_signal, class_idx, label=label)

    # retrain autoencoder
    for _ in range(ae_steps_per_frame):
        batch = live_dataset.get_random_human_sigs(ae_batch_size)
        if batch is not None:
            loss_val = ae_trainer.train_step(batch)

    # retrain OKM
    all_latents = live_dataset.get_random_latents(live_dataset.total_size())
    region_classifier.retrain(all_latents)

    # only recount labels when user is actively labeling — prevents unlabeled
    # data from aging out labeled entries and turning assignments into '?'
    if label_key_pressed:
        label_map = centroid_labeler.get_label_map(live_dataset)
        confidence_map = centroid_labeler.get_confidence_map(live_dataset)

    # visualise
    true_visualizer.update(human_signal.detach().cpu())
    with torch.no_grad():
        decoded = autoenc(human_signal.unsqueeze(0)).squeeze(0)
    decoded_visualizer.update(decoded.detach().cpu())

    clear_output(wait=True)
    current_label_name = label_keys[label_map[class_idx]] if label_map[class_idx] is not None else '?'
    label_display = ' | '.join(
        f"C{i}→{label_keys[label_map[i]] if label_map[i] is not None else '?'} ({confidence_map[i]:.0%})"
        for i in range(okm_num_centroids)
    )
    print(f"Holding: {label_keys[label] if label is not None else None} | Loss: {loss_val:.6f} | Queues: {live_dataset.get_queue_sizes()}")
    print(f"Now:     C{class_idx} → {current_label_name} ({confidence_map[class_idx]:.0%})")
    print(f"Labels:  {label_display}")
    print(f"Auto:    {'ON  key=' + (active_key or 'none') if autopress_enabled else 'OFF'}")

    # autopress (spacebar toggles)
    # uses keyboard.press/release — held edge-triggered so the browser sees a
    # real keydown held until the gesture changes, not a flood of repeated events
    toggle_is_pressed = keyboard.is_pressed(autopress_toggle_key)
    if toggle_is_pressed and not toggle_was_pressed:
        autopress_enabled = not autopress_enabled
        if not autopress_enabled and active_key is not None:
            keyboard.release(active_key)
            active_key = None
    toggle_was_pressed = toggle_is_pressed
    if autopress_enabled:
        mapped_idx = label_map[class_idx]
        mapped_label = label_keys[mapped_idx] if mapped_idx is not None else None
        target_key = action_map.get(mapped_label) if mapped_label is not None else None

        if target_key != active_key:
            # release old key, press new one — only fires on transition
            if active_key is not None:
                keyboard.release(active_key)
            active_key = target_key
            if active_key is not None:
                keyboard.press(active_key)

# ensure no keys are left held on exit
if active_key is not None:
    keyboard.release(active_key)

Holding: None | Loss: 0.000016 | Queues: [200, 200, 198, 200, 200]
Now:     C2 → a (46%)
Labels:  C0→w (87%) | C1→d (60%) | C2→a (46%) | C3→s (87%) | C4→shift (64%)
Auto:    ON  key=none
